In [ ]:
import simpy
import numpy as np
import matplotlib.pyplot as plt

# define a class that will represent this checkout process and enable its simulation
class ServerSystem:
    # define the constructor which is a method used to initialize objects of this class
    def __init__(self, sim_time, num_servers, arrival_rate, service_rate, queue_constraint):
        ## Simulation parameters  ##
        # passing in first the instance of the class "self" by convention
        # then passing in the relevant characteristics of this process
        #
        # assigning the parameters to the objects attributes 
        self.sim_time = sim_time
        self.num_servers = num_servers
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate
        self.queue_constraint = queue_constraint 

        ## Simpy  ##
        #
        # create a simpy environment which manages the simulation time and processing of events
        self.env = simpy.Environment()

        # create resources and bind to the environment that was created
        self.server = simpy.Resource(self.env, capacity=num_servers)

        ## process metrics ##
        # this is essentially the process accounting using lists
        #
        # Time each customer spends in the system, from arrival to departure
        self.flow_time = []
        # Time each customer spends in the queue, from arrival until service begins
        self.wait_time = []
        # Counter for number of customers who have completed service
        self.finished_customers = 0
        # Counter for number of customers who arrived
        self.arrived_customers = 0 

        # Times at which monitoring inventory occurs
        self.inv_time = []
        # Inventory in queue at monitoring times
        self.inv_queue = []
        # Inventory in service at monitoring times
        self.inv_service =[]
        # Inventory in system at monitoring times
        self.inv_system = []

    def monitor(self):
        """
        Record queue and WIP inventories every minute
        """
        while True:
            # update the accounting logs
            current_time = self.env.now 
            queue = len(self.server.queue)
            in_service = len(self.server.users)
            self.inv_time.append(current_time)
            self.inv_queue.append(queue)
            self.inv_service.append(in_service)
            self.inv_system.append(queue + in_service)
            # yield means process is suspended until the event occurs
            # here, 1 minute later
            yield self.env.timeout(1) 


    def customer(self, arrival_time):
        """
        Model the customer flow process
        """
        # the customer asks for the server

        if len(self.server.queue) < self.queue_constraint: 
            with self.server.request() as request:
                # wait until the server is available
                yield request
                # log how long it took to get the server
                self.wait_time.append(self.env.now - arrival_time)
                # now simulate a random wait time, here using the exponential distribution
                # wait for that to complete via a yield
                yield self.env.timeout(np.random.exponential(1/self.service_rate))
                # customer is finished so increment the finished customer counter
                self.finished_customers += 1
                # record how long their total flow time was
                self.flow_time.append(self.env.now - arrival_time)


    def gen_arrivals(self):
        """
        Customer generation process for arrivals
        """
        # simulate the arrivals of customers
        # here we wait a random exponential amount of time and then 
        # create a process for that customer and pass their arrival time in
        while True:
            yield self.env.timeout(np.random.exponential(1/self.arrival_rate))

            self.arrived_customers += 1 

            self.env.process(self.customer(self.env.now))


    def simulate(self):
        """
        Runs the simulation
        """
        # start each of these processes and then run the simulation for a desired amount of time
        self.env.process(self.monitor())
        self.env.process(self.gen_arrivals())
        self.env.run(until=self.sim_time)

# total simulation time in minutes
SIM_TIME = 100000

# number of servers: 14
NUM_SERVERS = 14

# mean customer arrival rate: 2 customers/min
LAMBDA = 2

# mean service rate: 0.20 customers/min
MU = 0.20

# Maximum number of customers allowed to wait in the queue (Buffer Capacity K)
# If the queue length reaches this number, new arrivals will be blocked
QUEUE_CONSTRAINT = 2 


# Instantiate the class and run it
system = ServerSystem(SIM_TIME, NUM_SERVERS, LAMBDA, MU, QUEUE_CONSTRAINT) 
system.simulate()

In [2]:
fraction_throughput_lost = 1 - system.finished_customers / system.arrived_customers
print(f"Fraction of Throughput Lost: {fraction_throughput_lost:.4f}")

Fraction of Throughput Lost: 0.0271


In [3]:
# Average Utilization (r)
# Throughput * Avg_Service_Time / c

throughput = system.finished_customers / SIM_TIME
avg_service_time = 1 / MU
utilization = (throughput * avg_service_time) / NUM_SERVERS
print(f"Average Utilization (r) : {utilization:.4f}")

Average Utilization (r) : 0.6959


In [4]:
# Probability of blocking (Pb)
# 1 - (Finished / Arrived)

prob_blocking = 1 - (system.finished_customers / system.arrived_customers)
print(f"Probability of blocking (Pb) : {prob_blocking:.4f}")

Probability of blocking (Pb) : 0.0271


In [5]:
# Probability of waiting (if not blocked) (Pw)
# (Number of people who waited / Total number of people entered)
wait_times = np.array(system.wait_time)
num_entered = system.finished_customers # Equivalent to number of people entered
num_waited = np.sum(wait_times > 0)     # Count customers with wait_time > 0

prob_waiting = num_waited / num_entered
print(f"Probability of waiting (Pw): {prob_waiting:.4f}")

Probability of waiting (Pw): 0.0972


In [6]:
# Average queue length (Ii)
# Mean of Monitor results (Average Queue Length)

avg_queue_len = np.mean(system.inv_queue)

print(f"Average queue length (Ii): {avg_queue_len:.4f}")

Average queue length (Ii): 0.0940


In [7]:
# Overall Average wait (Ti)
# Average wait time of all entered customers (including those who didn't wait)

avg_wait = np.mean(wait_times)

print(f"Overall Average wait (Ti): {avg_wait:.4f}")

Overall Average wait (Ti): 0.0485


In [8]:
# Average flow time (T)
# Wait + Service

avg_flow_time = np.mean(system.flow_time)

print(f"Average flow time (T): {avg_flow_time:.4f}")


Average flow time (T): 5.0631


In [9]:
# Average Inventory (I)
# Mean of Monitor results (Average Inventory in System)

avg_inventory = np.mean(system.inv_system)

print(f"Average Inventory (I): {avg_inventory:.4f}")

Average Inventory (I): 9.8642


In [10]:
# Little’s Law consistency check:
# Monitor inventory is a snapshot-based time average.
# Little’s Law uses realized throughput and average flow time from event timestamps.
# A tiny difference is expected due to snapshot resolution and finite horizon effects.

throughput = system.finished_customers / SIM_TIME
avg_flow_time = np.mean(system.flow_time)

inventory_by_littles_law = throughput * avg_flow_time

print(f"Average Inventory (Little's Law): {inventory_by_littles_law:.4f}")
print(f"Average Inventory (Monitor Result): {avg_inventory:.4f}")

Average Inventory (Little's Law): 9.8649
Average Inventory (Monitor Result): 9.8642
